# Leak-free Prefix-Complement — train + reconstruction test

**Settings:** T4 GPU, Internet On.

Tests the core of the thesis (train to capture 'what B adds' with NO labels):
1. **Leak test** — perturbing a future B token must not change earlier predictions.
2. **Reconstruction** — how well B is rebuilt from A + causal edge (val perplexity, token acc).
3. **Recon-GAIN** — perplexity(with edge) vs perplexity(A-only). Big gain => the edge
   carries real, leak-free info about B that A lacks.

Data = IteraTeR clean insertions (HuggingFace). Self-supervised: the inserted phrase
is NEVER a training target.

In [ ]:
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/complement_wiki"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"
MAX_EXAMPLES    = 120000
UPLOAD_TO_DRIVE = True

In [ ]:
# 1. GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> T4 GPU")
if torch.cuda.get_device_properties(0).major < 7:
    raise RuntimeError("GPU is P100 - change to T4")
print(torch.cuda.get_device_name(0), "OK")

In [ ]:
# 2. Clone / pull + deps
import os
root = f"/kaggle/working/{REPO_NAME}"
os.system(f"git clone {REPO_URL} {root}" if not os.path.isdir(root) else f"cd {root} && git pull")
os.chdir(WORK_DIR)
os.system("pip install -q 'transformers>=4.35.0' 'datasets>=2.14.0' gdown")
print("cwd:", os.getcwd(), "| deps ready")

In [ ]:
# 3. Prefetch IteraTeR clean insertions (HuggingFace; cached)
import sys, os
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval_v3")
sys.path.insert(0, f"{WORK_DIR}/../retrieval_v2")
import data_wikiedits as dw
trips = dw.load_triples(max_examples=MAX_EXAMPLES + 4000, cache=True)
print("clean-insertion triples:", len(trips))
for t in trips[:3]:
    print("  +", repr(t["inserted"]))

In [ ]:
# 4. LEAK TEST (untrained is fine; this checks the masking is correct)
import os
os.chdir(WORK_DIR)
os.system("python test_leak.py")

In [ ]:
# 5. Smoke test
import os
os.chdir(WORK_DIR)
os.system("python train_prefix.py --smoke")

In [ ]:
# 6. Full training (reports perplexity + recon-GAIN + leak check)
import os
os.chdir(WORK_DIR)
log = "/kaggle/working/prefix_train.log"
os.system(f"python train_prefix.py --max_examples {MAX_EXAMPLES} 2>&1 | tee {log}")
print("\ndone")

In [ ]:
# 7. Verify + optional rclone upload of checkpoint
import os, shutil
best = f"{WORK_DIR}/models/prefix_complement_best.pt"
if os.path.exists(best):
    print("checkpoint:", os.path.getsize(best)/1e6, "MB")
    shutil.copy(best, "/kaggle/working/prefix_complement_best.pt")
else:
    print("NOT FOUND - check log")
if UPLOAD_TO_DRIVE and os.path.exists(best):
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret("RCLONE_CONF")
        os.makedirs("/root/.config/rclone", exist_ok=True)
        open("/root/.config/rclone/rclone.conf", "w").write(conf)
        os.system("curl -s https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/r.zip")
        os.system("cd /tmp && unzip -qo r.zip && cp rclone-*-linux-amd64/rclone /usr/bin/ && chmod +x /usr/bin/rclone")
        rc = os.system(f"rclone copy {best} gdrive: --drive-root-folder-id {DRIVE_FOLDER_ID} -P")
        print("uploaded" if rc == 0 else f"rclone exit {rc}")
    except Exception as e:
        print("upload skipped:", e)

---
**Read the final report (cell 6 output):**
- `leak-free` must say LEAK-FREE (before-perturb diff ~0).
- `recon-GAIN` > 0 means the edge genuinely helps reconstruct B beyond A.
- Low val perplexity + decent token_acc = good reconstruction.

Next (separate step): the **recovery eval** — does the edge match the inserted phrase
better than baselines (the thesis test).